In [1]:
# 1. Pandas & Numpy (For handling our dataset and numbers)
import pandas as pd
import numpy as np

# 2. NLTK & Regular Expressions (For cleaning our messy SMS text)
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# 3. Scikit-Learn tools (For splitting data and TF-IDF)
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# 4. Matplotlib & Seaborn (For drawing graphs and Confusion Matrices later)
import matplotlib.pyplot as plt
import seaborn as sns

# Download the specific NLTK dictionaries we need for text cleaning
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

print("All libraries imported successfully!")

All libraries imported successfully!


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\aasra\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\aasra\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\aasra\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [2]:
#2.1
# 1. Load the dataset 
# (We use 'latin-1' encoding because the Kaggle dataset contains some special text characters)
try:
    df = pd.read_csv('spam.csv', encoding='latin-1')
    
    # 2. Drop the 3 empty/useless columns that Kaggle includes by mistake
    df = df.drop(['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], axis=1)
    
    # 3. Rename the remaining columns to something simple
    df.columns = ['label', 'text']
    
    print("Dataset loaded successfully! Here are the first 5 rows:")
    
    # 4. Display the first 5 rows of our dataset
    display(df.head())

except FileNotFoundError:
    print("ERROR: 'spam.csv' not found! Please make sure the file is in the exact same D:\\SMS_Spam_Project folder.")

Dataset loaded successfully! Here are the first 5 rows:


,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [3]:
#2.2
# 1. Initialize our text-cleaning tools
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# 2. Create the "Washing Machine" function
def clean_text(message):
    # Step A: Remove everything that is not a letter (like punctuation, numbers, links)
    message = re.sub('[^a-zA-Z]', ' ', message)
    
    # Step B: Convert to lowercase
    message = message.lower()
    
    # Step C: Split the sentence into individual words
    words = message.split()
    
    # Step D: Remove stop-words and lemmatize the remaining words
    cleaned_words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    
    # Step E: Join the words back together into a single cleaned string
    return ' '.join(cleaned_words)

# 3. Apply our function to every single message in the dataset
df['cleaned_text'] = df['text'].apply(clean_text)

# 4. Convert 'ham' and 'spam' into 0 and 1
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

print("Step 2.2 Complete! Here is the before-and-after comparison:")

# 5. Display the results to prove it worked
display(df[['label', 'label_num', 'text', 'cleaned_text']].head())

Step 2.2 Complete! Here is the before-and-after comparison:


,label,label_num,text,cleaned_text
0,ham,0,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available bugis n great ...
1,ham,0,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,spam,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entry wkly comp win fa cup final tkts st ...
3,ham,0,U dun say so early hor... U c already then say...,u dun say early hor u c already say
4,ham,0,"Nah I don't think he goes to usf, he lives aro...",nah think go usf life around though


In [4]:
## 3
# 1. Define our Features (X) and Target Labels (y)
X = df['cleaned_text']
y = df['label_num']

# 2. Split the data into 80% Training and 20% Testing
# (random_state=42 ensures we get the exact same random shuffle every time we run the code)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Data Split Complete!")
print(f"We have {X_train.shape[0]} messages for training and {X_test.shape[0]} messages for testing.\n")

# 3. Initialize the TF-IDF Vectorizer
# (max_features=3000 keeps only the top 3000 most important words, keeping our models lightweight!)
tfidf = TfidfVectorizer(max_features=3000)

# 4. Apply TF-IDF to the training data (The model LEARNS the vocabulary here)
X_train_tfidf = tfidf.fit_transform(X_train)

# 5. Apply TF-IDF to the testing data (The model ONLY translates here, it cannot learn new words from the test)
X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF Vectorization Complete!")
print(f"Training Matrix Shape: {X_train_tfidf.shape} -> (Messages, Words)")
print(f"Testing Matrix Shape:  {X_test_tfidf.shape}  -> (Messages, Words)")

Data Split Complete!
We have 4457 messages for training and 1115 messages for testing.

TF-IDF Vectorization Complete!
Training Matrix Shape: (4457, 3000) -> (Messages, Words)
Testing Matrix Shape:  (1115, 3000)  -> (Messages, Words)
